In [89]:
# Load the training data
import numpy as np
import pandas as pd
import math
import torch
from matplotlib import pyplot as plt
from torch import nn, optim
from time import time

df = pd.read_csv('data/colorings29.csv', sep=',') # edge-colorings data for configuration 30 in U_2822.conf
column_names = [c for c in df.columns]
del column_names[-1] # the last column is the output column, delete its name from the list
labels = torch.tensor(df['Y'].tolist()) # we store the output column separately
num_samples = len(labels) # 88574 for this dataset

# convert the dataframe to a pytorch tensor
inputs = torch.tensor([[df[column_names[j]][k] for j in range(36)] for k in range(num_samples)], dtype=torch.float32)

In [101]:
# take a look at the range of values in the sample labels and their distribution
values_dict={}
for j in range(num_samples):
    values_dict[labels[j].item()]=values_dict.get(labels[j].item(),0)+1
[(j,values_dict[j]) for j in range(len(values_dict))]

[(0, 4024),
 (1, 68354),
 (2, 1214),
 (3, 963),
 (4, 954),
 (5, 927),
 (6, 978),
 (7, 1167),
 (8, 1354),
 (9, 1573),
 (10, 1936),
 (11, 2278),
 (12, 1872),
 (13, 922),
 (14, 58)]

In [133]:
# separate the dataset into a training set and a validation set
torch.manual_seed(0) # to get the same result each time we run the code
permutation = torch.randperm(num_samples)
num_train_samples = int(num_samples * 0.8)
num_val_samples = num_samples - num_train_samples

# these are the samples we will use for the training
traininputs = inputs[permutation[range(num_train_samples)]]
trainlabels = labels[permutation[range(num_train_samples)]]

# these are the validation samples
valinputs = inputs[permutation[range(num_train_samples,num_samples)]]
vallabels = labels[permutation[range(num_train_samples,num_samples)]]

In [184]:
# Build the neural network

input_size = 36
hidden_sizes = [64,32,12,12]
output_size = 15

# hyperparameters
loss_function = nn.NLLLoss() # this is the loss function we'll be using
learning_rate = 0.1
batch_learning_rate = 0.001
use_batches = True # controls whether we are training in batches or using the entire dataset
batch_size = 1024

loss_history = []

reading_saved_model = False
saved_model_path1 = 'savedmodel1.pth'
saved_model_path2 = 'savedmodel2.pth'

model = nn.Sequential(nn.Linear(input_size, hidden_sizes[0]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[0], hidden_sizes[1]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[1], hidden_sizes[2]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[2], output_size),
                      nn.LogSoftmax(dim=1))


if reading_saved_model:
    # first way of doing it
    # for this method, we don't even need to have defined model earlier as we did in the code above
    # model = torch.load(saved_model_path1)
    # print(f"read model from saved file '{saved_model_path1}'")

    # second way of doing it
    model.load_state_dict(torch.load(saved_model_path2))
    print(f"read model from saved file '{saved_model_path2}'")
    
print(model)
print("number of parameters =",sum(p.numel() for p in model.parameters() if p.requires_grad))

Sequential(
  (0): Linear(in_features=36, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=12, bias=True)
  (5): ReLU()
  (6): Linear(in_features=12, out_features=15, bias=True)
  (7): LogSoftmax(dim=1)
)
number of parameters = 5039


In [189]:
# Training process

optimizer = optim.SGD(model.parameters(), lr=batch_learning_rate if use_batches else learning_rate, momentum=0.9)
time0 = time()
epochs = 10000

for e in range(epochs):
    optimizer.zero_grad()

    if not use_batches:
        # code for training without batches:
        training_output = model(traininputs) # forward pass
        loss = loss_function(training_output, trainlabels) # compute the loss
        loss.backward() # backward pass
        optimizer.step() # adjust the parameters
    else:
        # code for training with batches:
        permutation = torch.randperm(num_train_samples)
        running_loss = 0
        num_batches = 0
        for i in range(0,num_train_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = traininputs[indices], trainlabels[indices]
            batch_output = model(batch_x) # forward pass
            batch_loss = loss_function(batch_output, batch_y) # compute the loss
            batch_loss.backward() # backward pass
            optimizer.step() # adjust the parameters
            num_batches += 1
            running_loss += batch_loss
        loss = running_loss / num_batches
    
    loss_history.append(loss.item()) # keep track of how the loss is evolving for doing plots etc
    
    if e%100==0:
        print(f"Epoch {e}: loss={loss:.5}")

trainoutput = model(traininputs) # do a final forward pass over the training set to compute classification accuracy
# accuracy = sum([1 if ((output[j][0].item() > output[j][1].item() and labels[j]==0) or (output[j][0].item() <= output[j][1].item() and labels[j]==1)) else 0 for j in range(num_samples)])
#         
# print(f"\nTraining Time (in seconds)={time()-time0:.4}, accuracy={accuracy}/{num_samples} = {100*accuracy/num_samples:.4}%")

Epoch 0: loss=1.1003
Epoch 100: loss=1.1005
Epoch 200: loss=1.0998
Epoch 300: loss=1.1014
Epoch 400: loss=1.1
Epoch 500: loss=1.0996
Epoch 600: loss=1.1015
Epoch 700: loss=1.1013
Epoch 800: loss=1.0998
Epoch 900: loss=1.1005
Epoch 1000: loss=1.1006
Epoch 1100: loss=1.1004
Epoch 1200: loss=1.099
Epoch 1300: loss=1.0995
Epoch 1400: loss=1.0984
Epoch 1500: loss=1.0987
Epoch 1600: loss=1.1008
Epoch 1700: loss=1.0999
Epoch 1800: loss=1.0989
Epoch 1900: loss=1.0998
Epoch 2000: loss=1.1017
Epoch 2100: loss=1.0984
Epoch 2200: loss=1.0986
Epoch 2300: loss=1.1014
Epoch 2400: loss=1.1018
Epoch 2500: loss=1.1002
Epoch 2600: loss=1.0999
Epoch 2700: loss=1.1019
Epoch 2800: loss=1.1001
Epoch 2900: loss=1.0996
Epoch 3000: loss=1.0976
Epoch 3100: loss=1.1022
Epoch 3200: loss=1.1004
Epoch 3300: loss=1.1012
Epoch 3400: loss=1.0998
Epoch 3500: loss=1.101
Epoch 3600: loss=1.1004
Epoch 3700: loss=1.1001
Epoch 3800: loss=1.0997
Epoch 3900: loss=1.1004
Epoch 4000: loss=1.1032
Epoch 4100: loss=1.0987
Epoch 420

In [179]:
trainoutput = model(traininputs)
trainpredictions=[torch.argmax(trainoutput[j]).item() for j in range(len(trainoutput))]
correcttrainpredictions=sum([1 if trainpredictions[j]==trainlabels[j].item() else 0 for j in range(len(trainoutput))])

valoutput = model(valinputs)
valpredictions=[torch.argmax(valoutput[j]).item() for j in range(len(valoutput))]
correctvalpredictions=sum([1 if valpredictions[j]==vallabels[j].item() else 0 for j in range(len(valoutput))])

In [180]:
from collections import Counter
Counter(trainpredictions)

Counter({1: 70851, 0: 8})

In [181]:
Counter(valpredictions)

Counter({1: 17713, 0: 2})

In [182]:
correctvalpredictions

13701

In [183]:
correcttrainpredictions

54649

In [188]:
batch_learning_rate = 0.00005